In [6]:
!pip install -q langchain langchain-community langchain-core langchain-text-splitters faiss-cpu sentence-transformers pypdf groq

import os
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import HuggingFaceEmbeddings
from groq import Groq
os.environ["GROQ_API_KEY"] = "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"
client = Groq(api_key=os.environ["GROQ_API_KEY"])
pdf_path = r"C:\Users\hp\Downloads\DeepLearning_Notes.pdf"   
loader = PyPDFLoader(pdf_path)
docs = loader.load()
print("\nTotal pages loaded:", len(docs))
print("Sample text:\n", docs[0].page_content[:300])
splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=150
)
chunks = splitter.split_documents(docs)
print("\nTotal chunks created:", len(chunks))
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)
db = FAISS.from_documents(chunks, embeddings)
print("\nFAISS index created successfully!")
def debug_context(query):
    docs = db.similarity_search(query, k=3)
    print("\n================ RETRIEVED CONTEXT ================\n")
    for i, d in enumerate(docs):
        print(f"--- CHUNK {i+1} ---")
        print(d.page_content[:500])
        print("-" * 60)
STRICT_PROMPT = """
You are a document QA assistant.
Rules:
- Use ONLY the provided context.
- If answer is not present in context, say:
  I could not find this
- If context contains relevant info, answer clearly and briefly.
"""
def get_context(query):
    docs = db.similarity_search(query, k=3)
    return "\n\n".join([d.page_content for d in docs])
def ask(query):
    context = get_context(query)
    messages = [
        {"role": "system", "content": STRICT_PROMPT},
        {"role": "user", "content": f"Context:\n{context}\n\nQuestion: {query}"}
    ]
    response = client.chat.completions.create(
        model="llama-3.1-8b-instant",
        messages=messages
    )
    return response.choices[0].message.content
in_scope = [
    "What is the main topic of the document?",
    "What are the key points mentioned in the notes?",
    "Define the main concept explained in the PDF."
]
out_scope = [
    "Who won the FIFA World Cup 2022?",
    "What is the capital of France?"
]
print("\n\n🔍 DEBUG CHECK (RUNNING RETRIEVAL TEST)\n")
debug_context(in_scope[0])
print("\n================ IN-SCOPE QUESTIONS ================\n")
for q in in_scope:
    ans = ask(q)
    print("Q:", q)
    print("A:", ans)
    print("-" * 80)
print("\n================ OUT-OF-SCOPE QUESTIONS ================\n")
for q in out_scope:
    ans = ask(q)
    print("Q:", q)
    print("A:", ans)
    if "could not find this" in ans.lower():
        print("✅ PASS (Correct refusal)")
    else:
        print("❌ FAIL (Hallucination or wrong behavior)")

    print("-" * 80)


Total pages loaded: 16
Sample text:
 AI / ML INTERNSHIP PROGRAM
WEEK 3 — STUDENT NOTES
Deep Learning Foundations & Generative AI
■ 8 Topics  Code Examples  Exercises  Day 1 of Week 3
Duration
Day 1 of 4 | ~3 hrs teaching + 2 hrs hands-on coding
Platform
Google Colab (recommended) or VS Code + Jupyter
Week Goal
Understand Neural Netw

Total chunks created: 28


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]


FAISS index created successfully!


🔍 DEBUG CHECK (RUNNING RETRIEVAL TEST)


================ RETRIEVED CONTEXT ================

--- CHUNK 1 ---
1
What is a Neural Network?
A neural network is a system of connected layers — each layer learns a different level of abstraction from data.
It is loosely inspired by how neurons in the brain connect and signal each other. You already know that Deep
Learning uses neural networks — now we will understand exactly how they work.
The Structure — Input → Hidden → Output
Every neural network has three types of layers:
Layer
What it does
Example (predicting house price)
Input Layer
Receives the raw fea
------------------------------------------------------------
--- CHUNK 2 ---
AI / ML INTERNSHIP PROGRAM
WEEK 3 — STUDENT NOTES
Deep Learning Foundations & Generative AI
■ 8 Topics  Code Examples  Exercises  Day 1 of Week 3
Duration
Day 1 of 4 | ~3 hrs teaching + 2 hrs hands-on coding
Platform
Google Colab (recommended) or VS Code + Jupyter
Week Go